In [ ]:
# INF201 - Deliverable 2
# Task 1:  File ingestion and regular expressions

from pathlib import Path
import re

# Compiled once at module level
LOG_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>POS_\d+)\] "
    r"item=(?P<item>\w+) "
    r"price=(?P<price>\d+\.\d{2}) "
    r"(?P<currency>\w{3}) "
    r"\((?P<status>\w+)\)"
)


def filereader(file_path: str | Path) -> tuple[str, str]:
    """Reading a text file and it detects if it's UTF-8 or Latin-1
    
    UTF-8 is the first one to get tried, because that one is more
    strict and raises an error on invalid byte sequence. Latin-1
    on the other hand, accepts every byte, so it can never fail,
    so its used after
    
    Arguments: 
            file_path: Is a path to file, as a string or Path object
    Returns: 
            A tuple (file_text_content, encoding_used)
    """
    path = Path(file_path)
    try:
        file_text_content = path.read_text(encoding="utf-8")
        encoding_used = "utf-8"
    except UnicodeDecodeError:
        file_text_content = path.read_text(encoding="latin-1")
        encoding_used = "latin-1"
    return file_text_content, encoding_used


def log_parser(text: str) -> list[dict[str, str | float]]:
    """This function extracs the transaction lines from log text and 
    ignores system noise. Lines that dont match the sale lines like
    (printer, barcode, network messages) are skipped.
    
    Arguments: 
            text: The content of of the log file
    Returns: 
            A list of dictionaries with the keys, timestamp, register_id,
            item, price, currecny and status.
    """
    records = []
    for line in text.splitlines():
        transaction = LOG_PATTERN.match(line)
        if transaction is not None:
            dict_mode = transaction.groupdict()
            dict_mode["price"] = float(dict_mode["price"])
            records.append(dict_mode)
    return records
